# OpsPilot Real RAG Backend\nRun all cells to start the FastAPI server with Vector Search!

In [ ]:
!pip install fastapi uvicorn networkx pyyaml pydantic markdown nest-asyncio langchain langchain-community chromadb sentence-transformers openai

In [ ]:
import os
from dotenv import load_dotenv
import yaml
from pathlib import Path
import networkx as nx
import random
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio

load_dotenv()

# RAG specific imports
from langchain_text_splitters import MarkdownHeaderTextSplitter
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# Apply nest_asyncio so uvicorn can run inside the Jupyter event loop
nest_asyncio.apply()

RUNBOOKS_DIR = Path("runbooks")


## 1. Runbook Loader

In [ ]:
def load_runbooks():
    runbooks = []
    if not RUNBOOKS_DIR.exists(): return runbooks

    for filepath in RUNBOOKS_DIR.glob("*.md"):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
            
        if content.startswith("---"):
            try:
                parts = content.split("---", 2)
                if len(parts) >= 3:
                    frontmatter = yaml.safe_load(parts[1])
                    body = parts[2].strip()
                    runbooks.append({
                        "slug": filepath.stem,
                        "title": frontmatter.get("title", filepath.stem),
                        "category": frontmatter.get("category", "General"),
                        "severity": frontmatter.get("severity", "Medium"),
                        "tags": frontmatter.get("tags", []),
                        "links": frontmatter.get("links", []),
                        "content": body
                    })
            except Exception as e:
                print(f"Error parsing {filepath}: {e}")
    return runbooks

runbooks_data = load_runbooks()
print(f"Loaded {len(runbooks_data)} runbooks.")


## 2. Graph Builder

In [ ]:
def build_graph(runbooks):
    G = nx.DiGraph()
    for rb in runbooks:
        G.add_node(rb["slug"], title=rb["title"], category=rb["category"])
        
    for rb in runbooks:
        for link in rb.get("links", []):
            if G.has_node(link):
                G.add_edge(rb["slug"], link)
                
    nodes = [{"id": n, "title": d.get("title", n), "category": d.get("category", "General")} for n, d in G.nodes(data=True)]
    edges = [{"source": u, "target": v} for u, v in G.edges()]
    return {"nodes": nodes, "edges": edges}


## 3. RAG Pipeline Setup (ChromaDB + Embeddings)

In [ ]:
# Initialize ChromaDB in-memory client
chroma_client = chromadb.Client()

# Use local sentence-transformers (downloads a fast ~90MB model automatically)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.create_collection(name="runbooks", embedding_function=ef)

# Chunk the markdown
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

documents = []
metadatas = []
ids = []

chunk_id = 0
for rb in runbooks_data:
    splits = markdown_splitter.split_text(rb["content"])
    for split in splits:
        documents.append(split.page_content)
        
        # Merge runbook metadata with chunk metadata
        meta = {
            "source_runbook": rb["title"],
            "severity": rb["severity"]
        }
        for k, v in split.metadata.items():
            meta[k] = str(v)
            
        metadatas.append(meta)
        ids.append(f"chunk_{chunk_id}")
        chunk_id += 1

if documents:
    collection.add(
        documents=documents,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Successfully chunked and embedded {len(documents)} snippets into ChromaDB!")


## 4. RAG Chat Logic (Retrieval + Generation)

In [ ]:


def generate_chat_response(question, runbooks):
    # 1. RETRIEVAL (Vector Search)
    results = collection.query(
        query_texts=[question],
        n_results=3
    )
    
    if not results["documents"] or not results["documents"][0]:
        return {
            "answer": "I don't have any runbooks matching this question.",
            "confident": False,
            "source_chunks": [],
            "expanded_via_graph": []
        }
        
    retrieved_chunks = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]
    
    source_chunks = []
    expanded_via_graph = []
    context_text = ""
    
    for i, chunk in enumerate(retrieved_chunks):
        meta = retrieved_metadata[i]
        rb_title = meta.get("source_runbook", "Unknown Runbook")
        section = meta.get("Header 2", "")
        
        source_chunks.append(f"{rb_title} ({section})")
        context_text += f"\n--- RUNBOOK: {rb_title} SECTION: {section} ---\n{chunk}\n"
        
        # Grab linked graph runbooks for expansion
        for rb in runbooks:
            if rb["title"] == rb_title and rb.get("links"):
                for link in rb["links"]:
                    expanded_via_graph.append(f"Linked: {link}")

    # Deduplicate expanded links
    expanded_via_graph = list(set(expanded_via_graph))

    # 2. GENERATION (OpenAI or Fallback)
    client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
    
    prompt = f"""
    You are OpsPilot, an on-call AI assistant. Answer the user's infrastructure question based ONLY on the provided runbook context.
    Keep the answer concise and step-by-step.
    
    CONTEXT:
    {context_text}
    
    USER QUESTION: {question}
    """
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )
        answer = response.choices[0].message.content
        confident = True
    except Exception as e:
        # Graceful fallback if API key is invalid/placeholder
        answer = f"**[LLM Generation Failed - Invalid API Key]**\n\nI found {len(retrieved_chunks)} relevant runbook sections using Vector Search:\n"
        answer += context_text
        confident = False

    return {
        "answer": answer,
        "confident": confident,
        "source_chunks": source_chunks,
        "expanded_via_graph": expanded_via_graph,
        "graph_edges_used": []
    }


## 5. FastAPI Setup

In [ ]:
app = FastAPI(title="OpsPilot Real RAG Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    question: str

@app.get("/runbooks")
def get_runbooks():
    return [
        {"slug": rb["slug"], "title": rb["title"], "category": rb["category"], "severity": rb["severity"], "tags": rb["tags"]}
        for rb in runbooks_data
    ]

@app.get("/graph")
def get_graph():
    return build_graph(runbooks_data)

@app.post("/chat")
def chat(req: ChatRequest):
    return generate_chat_response(req.question, runbooks_data)


## 6. Run Server

In [ ]:
print("Starting True RAG server on port 8001...")
print("The frontend should be configured to point to http://localhost:8001")
print("Note: This cell will block execution while the server is running.")

import uvicorn

config = uvicorn.Config(app, host="127.0.0.1", port=8001)
server = uvicorn.Server(config)

await server.serve()